In [1]:
import os
os.environ["NPU_VISIBLE_DEVICES"]="6"
os.environ["ASCEND_RT_VISIBLE_DEVICES"]="6"
import json
from tqdm import tqdm
from functools import partial
from typing import Optional, Tuple, Union, Dict, List, Any

import torch
import torch.nn as nn
from transformers import AutoTokenizer
from datasets import load_dataset


from transformers.cache_utils import Cache
from transformers.utils import add_start_docstrings, ModelOutput, logging
from transformers.modeling_utils import PreTrainedModel
from transformers import LlamaModel, AutoConfig
from transformers.configuration_utils import PretrainedConfig

from typing import TYPE_CHECKING
from dataclasses import dataclass

logger = logging.get_logger(__name__)

if TYPE_CHECKING:
    from transformers import PreTrainedModel


/home/lihz/miniconda3/envs/workspace/lib/python3.10/site-packages/torch_npu/utils/path_manager.py:82: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/latest owner does not match the current user.
  warnings.warn(f"Warning: The {path} owner does not match the current user.")
/home/lihz/miniconda3/envs/workspace/lib/python3.10/site-packages/torch_npu/utils/path_manager.py:82: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/8.0.RC2/aarch64-linux/ascend_toolkit_install.info owner does not match the current user.
  warnings.warn(f"Warning: The {path} owner does not match the current user.")


In [2]:
base_model = "/data/pretrained-models/meta/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_model)

In [3]:
base_dataset = "/data/datasets/Llama-3.2-3B-Instruct-evals"
general_datasets = [
    "Llama-3.2-3B-Instruct-evals__mmlu__details",
    "/data/datasets/ultrachat_200k",
]
reason_datasets = [
    "Llama-3.2-3B-Instruct-evals__gpqa__details",
    "Llama-3.2-3B-Instruct-evals__arc_challenge__details"
]
math_datasets = [
    # "Llama-3.2-3B-Instruct-evals__gsm8k__details",
    "gsm8k",
    "/data/datasets/MathInstruct",
    "Llama-3.2-3B-Instruct-evals__math__details"
]
humaneval_datasets = [
    "evalplus/humanevalplus",
]
magicoder_datasets = [
    "/data/datasets/Magicoder-Evol-Instruct-110K",
]
mbpp_datasets = [
    "evalplus/mbppplus"
]

In [4]:
def preprocess_gsm8k(examples:Dict[str, Any], tokenizer:AutoTokenizer)->Dict[str, Any]:
    prefix = "Given the following problem, reason and give a final answer to the problem.\nProblem: {{question}}\nYour response should end with \"The final answer is [answer]\" where [answer] is the response to the problem.\n"
    icl = [
        {
            "role" : "user",
            "content" : "There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?"
        },
        {
            "role" : "assistant",
            "content" : "There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The final answer is 6"
        },
        {
            "role": "user",
            "content": "If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?"
        },
        {
            "role": "assistant",
            "content" : "There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The final answer is 5"
        },
        {
            "role": "user",
            "content" : "Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?",
        },
        {
            "role" : "assistant",
            "content" : "Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 74. After eating 35, they had 74 - 35 = 39. The final answer is 39"
        },
        {
            "role" : "user",
            "content" : "Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?"
        },
        {
            "role" : "assistant",
            "content" : "Jason started with 20 lollipops. Then he had 12 after giving some to Denny. So he gave Denny 20 - 12 = 8. The final answer is 8"
        },
        {
            "role" : "user",
            "content" : "Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?"
        },
        {
            "role" : "assistant",
            "content" : "Shawn started with 5 toys. If he got 2 toys each from his mom and dad, then that is 4 more toys. 5 + 4 = 9. The final answer is 9"
        },
        {
            "role" : "user",
            "content" : "There were nine computers in the server room. Five more computers were installed each day, from monday to thursday. How many computers are now in the server room?"
        },
        {
            "role" : "assistant",
            "content" : "There were originally 9 computers. For each of 4 days, 5 more computers were added. So 5 * 4 = 20 computers were added. 9 + 20 is 29. The final answer is 29"
        },
        {
            "role" : "user",
            "content" : "Michael had 58 golf balls. On tuesday, he lost 23 golf balls. On wednesday, he lost 2 more. How many golf balls did he have at the end of wednesday?"
        },
        {
            "role" : "assistant",
            "content" : "Michael started with 58 golf balls. After losing 23 on tuesday, he had 58 - 23 = 35. After losing 2 more, he had 35 - 2 = 33 golf balls. The final answer is 33"
        },
        {
            "role" : "user",
            "content" : "Olivia has $23. She bought five bagels for $3 each. How much money does she have left?"
        },
        {
            "role" : "assistant",
            "content" : "Olivia had 23 dollars. 5 bagels for 3 dollars each will be 5 x 3 = 15 dollars. So she has 23 - 15 dollars left. 23 - 15 is 8. The final answer is 8"
        }
    ]
    for i in range(len(icl)):
        if icl[i]['role'] == "user":
            icl[i]['content'] = prefix.replace("{{question}}", icl[i]['content'])
    icl.append({
        "role": "user",
        "content": prefix.replace("{{question}}", examples["question"].strip())
    })
    messages = tokenizer.apply_chat_template(icl, tokenize=False, add_generation_prompt=True)
    return { "inst": messages }

def preprocess_ultrachat(examples:Dict[str, Any], tokenizer:AutoTokenizer)->Dict[str,Any]:
    messages = examples['messages']
    messages = tokenizer.apply_chat_template(messages[:-1] if len(messages) > 1 else messages, tokenize=False, add_generation_prompt=True)
    return { "inst":messages }


def task_preprocess(example:Dict[str, str], tokenizer:AutoTokenizer, task:str="humaneval")->Dict[str, str]:
    if task == "humaneval":
        instruction_prefix = "Please provide a self-contained Python script that solves the following problem in a markdown code block:"
        response_prefix = "Below is a Python script with a self-contained function that solves the problem and passes corresponding tests:"
        # some random words which servcleaes as the splitter
        _MAGIC_SPLITTER_ = "-[[]]-this-is-really-our-highest-priority-[[]]-"
        task_prompt = f"""\
{instruction_prefix}
```
{example['prompt'].strip()}
```
"""
        response = f"""\
{response_prefix}
```python
{_MAGIC_SPLITTER_}
```
"""
        task_prompt = tokenizer.apply_chat_template(
            [
                {"role": "user", "content": task_prompt},
                {"role": "assistant", "content": response},
            ],
            tokenize=False,
        ).split(_MAGIC_SPLITTER_)[0]
        return {
            "inst": task_prompt,
        }
    elif task == "mbpp":
        instruction_prefix = "Please provide a self-contained Python script that solves the following problem in a markdown code block:"
        response_prefix = "Below is a Python script with a self-contained function that solves the problem and passes corresponding tests:"
        # some random words which servcleaes as the splitter
        _MAGIC_SPLITTER_ = "-[[]]-this-is-really-our-highest-priority-[[]]-"
        python_prefix = 'Write a python function to '
        func_prefix = 'Write a function to '
        if python_prefix in example['prompt']:
            prefix = python_prefix
        elif func_prefix in example['prompt']:
            prefix = func_prefix
        else:
            prefix = ""
        prompt = example['prompt'].replace(prefix, '').strip().capitalize()
        task_prompt = f"""\
{instruction_prefix}
```
{example['code'].split(":")[0].strip()}:
    \"\"\"
    {prompt}
    >>> {example['test_list'][0].replace("assert", "").strip()}
    True
    \"\"\"
```
"""
        response = f"""\
{response_prefix}
```python
{_MAGIC_SPLITTER_}
```
"""
        task_prompt = tokenizer.apply_chat_template(
            [
                {"role": "user", "content": task_prompt},
                {"role": "assistant", "content": response},
            ],
            tokenize=False,
        ).split(_MAGIC_SPLITTER_)[0]
        return {
            "inst": task_prompt,
        }
    elif task == "magicoder":
        return {
            "inst": f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{example['instruction'].strip()}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"
            }
    elif task == "mathinstruct":
        return {
            "inst": f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{example['instruction'].strip()}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"
        }

  

In [5]:
general_datasets = [
    # load_dataset(
    #     base_dataset,
    #     name=item,
    #     num_proc=8,
    # ).map(
    #     partial(preprocess_fn_general, tokenizer=tokenizer),
    #     num_proc=8,
    # )['latest'] for item  in general_datasets
    load_dataset(
        base_dataset,
        name=general_datasets[0],
        num_proc=8,
    )['latest'],
    load_dataset(
        general_datasets[1],
        num_proc=8,
    ).map(
        partial(preprocess_ultrachat, tokenizer=tokenizer),
        num_proc=8
    )['train_sft'],
]
reason_datasets = [
    load_dataset(
        base_dataset,
        name=item,
        num_proc=8,
    )['latest'] for item  in reason_datasets
]
math_datasets = [
    # load_dataset(
    #     base_dataset,
    #     name=math_datasets[0],
    #     num_proc=8,
    # )['latest'],
    load_dataset(
        math_datasets[0],
        "main",
        num_proc=8,
    )['train'].map(
        partial(preprocess_gsm8k, tokenizer=tokenizer),
        num_proc=8,
    ),
    load_dataset(
        math_datasets[1],
        num_proc=8,
    ).map(
        partial(task_preprocess, tokenizer=tokenizer, task="mathinstruct"),
        num_proc=8,
    )['train'],
    load_dataset(
        base_dataset,
        name=math_datasets[2],
        num_proc=8,
    )['latest']
]
humaneval_datasets = [
    load_dataset(
        item,
        num_proc=8,
    ).map(
        partial(task_preprocess, tokenizer=tokenizer, task="humaneval"),
        num_proc=8,
    )['test'] for item in humaneval_datasets
]
magicoder_datasets = [
    load_dataset(
        item,
        num_proc=8,
    ).map(
        partial(task_preprocess, tokenizer=tokenizer, task="magicoder"),
        num_proc=8,
        load_from_cache_file=False,
    )['train'] for item in magicoder_datasets
]
mbpp_datasets = [
    load_dataset(
        item,
        num_proc=8,
    ).map(
        partial(task_preprocess, tokenizer=tokenizer, task="mbpp"),
        num_proc=8,
        load_from_cache_file=False,
    )['test'] for item in mbpp_datasets
]

Map (num_proc=8):   0%|          | 0/111183 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/378 [00:00<?, ? examples/s]

In [6]:
mix_domains = []

start = 1000
samples = 2100

repeat = 1 if len(general_datasets[0]) >= samples else 0
for _ in range(repeat):
    for i, item in enumerate(general_datasets[0]):
        if i <= start:
            continue
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['input_final_prompts'][0],
            'task': 'mmlu',
            'task_label': 0,
            'label': 1,
            'outputs': item['input_correct_responses'][0]
        })

repeat = 1 if len(general_datasets[1]) >= samples else 0
for _ in range(repeat):
    for i, item in enumerate(general_datasets[1]):
        if i <= start:
            continue
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'ultrachat',
            'task_label': 1,
            'label': 0,
            'outputs': item['inst'],
        })

In [7]:
repeat = 1 if len(reason_datasets[0]) >= samples else 1
for _ in range(repeat):
    for i, item in enumerate(reason_datasets[0]):
        if i <= 400:
            continue
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['input_final_prompts'][0],
            'task': 'gpqa',
            'task_label': 2,
            'label': 1,
            'outputs': item['input_correct_responses'][0]
        })
repeat = 1 if len(reason_datasets[1]) >= samples else 1
for _ in range(repeat):
    for i, item in enumerate(reason_datasets[1]):
        if i <= 1000:
            continue
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['input_final_prompts'][0],
            'task': 'arc_c',
            'task_label': 3,
            'label': 1,
            'outputs': item['input_correct_responses'][0]
        })


In [8]:
repeat = 1 if len(math_datasets[0]) >= samples else 0
for _ in range(repeat):
    for i, item in enumerate(math_datasets[0]):
        if i <= start:
            continue
        if i == samples:
            break
        # mix_domains.append({
        #     'inputs': item['input_final_prompts'][0],
        #     'task': 'gsm8k',
        #     'task_label': 4,
        #     'label': 2,
        #     'outputs': item['input_correct_responses'][0]
        # })
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'gsm8k',
            'task_label': 4,
            'label': 2,
            'outputs': item['answer']
        })

# repeat = 1 if len(math_datasets[1]) >= samples else 1
# for _ in range(repeat):
#     for i, item in enumerate(math_datasets[1]):
#         if i <= 1000:
#             continue
#         if i == samples:
#             break
#         mix_domains.append({
#             'inputs': item['inst'],
#             'task': 'mathinstruct',
#             'task_label': 5,
#             'label': 2,
#             'outputs': item['output']
#         })

repeat = 1 if len(math_datasets[2]) >= samples else 0
for _ in range(repeat):
    for i, item in enumerate(math_datasets[2]):
        if i <= start:
            continue
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['input_final_prompts'][0],
            'task': 'math',
            'task_label': 5,
            'label': 2,
            'outputs': item['input_correct_responses'][0]
        })

In [9]:
repeat = 1 if len(humaneval_datasets[0]) >= samples else 1
for _ in range(repeat):
    for i, item in enumerate(humaneval_datasets[0]):
        if i <= 150:
            continue
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'humaneval',
            'task_label': 6,
            'label': 3,
            'outputs': item['canonical_solution']
        })

repeat = 1 if len(mbpp_datasets[0]) >= samples else 1
for _ in range(repeat):
    for i, item in enumerate(mbpp_datasets[0]):
        if i <= 300:
            continue
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'mbpp',
            'task_label': 7,
            'label': 3,
            'outputs': item['code']
        })

repeat = 1 if len(magicoder_datasets[0]) >= samples else 0
for _ in range(repeat):
    for i, item in enumerate(magicoder_datasets[0]):
        if i <= start:
            continue
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'magicoder',
            'task_label': 8,
            'label': 3,
            'outputs': item['response']
        })

In [10]:
print(len(mix_domains))

5796


In [11]:
with open("/data/lihz/datasets/mix_domains_eval/mix_domains_eval_v3.jsonl", 'w') as f:
    for item in mix_domains:
        f.write(json.dumps(item) + '\n')

In [12]:


@dataclass
class CLSOutput(ModelOutput):
    loss: Optional[Union[torch.FloatTensor, Dict[str, torch.FloatTensor]]] = None
    hidden_states: Optional[Union[Tuple[torch.FloatTensor, ...], Dict[str, torch.FloatTensor]]] = None
    activations: Optional[Union[Tuple[torch.FloatTensor, ...], Dict[str, torch.FloatTensor]]] = None

CLS_START_DOCSTRING = r"""
    This model inherits from [`PreTrainedModel`]. Check the superclass documentation for the generic methods the
    library implements for all its model (such as downloading or saving, resizing the input embeddings, pruning heads
    etc.)

    This model is also a PyTorch [torch.nn.Module](https://pytorch.org/docs/stable/nn.html#torch.nn.Module) subclass.
    Use it as a regular PyTorch Module and refer to the PyTorch documentation for all matter related to general usage
    and behavior.

    Parameters:
        config ([`CLSConfig`]):
            Model configuration class with all the parameters of the model. Initializing with a config file does not
            load the weights associated with the model, only the configuration. Check out the
            [`~PreTrainedModel.from_pretrained`] method to load the model weights.
"""

class CLSConfig(PretrainedConfig):
    r"""
    This is the configuration class to store the configuration of a [`CLSModel`]. It is used to instantiate an CLS
    model according to the specified arguments, defining the model architecture. Instantiating a configuration with the
    defaults will yield a similar configuration to that of the CLS-7B.

    Configuration objects inherit from [`PretrainedConfig`] and can be used to control the model outputs. Read the
    documentation from [`PretrainedConfig`] for more information.


    Args:
        vocab_size (`int`, *optional*, defaults to 32000):
            Vocabulary size of the CLS model. Defines the number of different tokens that can be represented by the
            `inputs_ids` passed when calling [`CLSModel`]
        hidden_size (`int`, *optional*, defaults to 4096):
            Dimension of the hidden representations.
        num_hidden_layers (`int`, *optional*, defaults to 32):
            Number of hidden layers in the Transformer decoder.
        hidden_act (`str` or `function`, *optional*, defaults to `"silu"`):
            The non-linear activation function (function or string) in the decoder.
        initializer_range (`float`, *optional*, defaults to 0.02):
            The standard deviation of the truncated_normal_initializer for initializing all weight matrices.

    ```python
    >>> from transformers import CLSModel, CLSConfig

    >>> # Initializing a CLS CLS-7b style configuration
    >>> configuration = CLSConfig()

    >>> # Initializing a model from the CLS-7b style configuration
    >>> model = CLSModel(configuration)

    >>> # Accessing the model configuration
    >>> configuration = model.config
    ```"""

    model_type = "cls"

    def __init__(
        self,
        hidden_size:int=4096,
        num_labels:int=4,
        num_tasks:int=8,
        initializer_range:float=0.02,
        **kwargs,
    ):
        super().__init__(
            **kwargs,
        )
        self.num_labels= num_labels
        self.num_tasks = num_tasks
        self.hidden_size = hidden_size
        self.initializer_range = initializer_range

@add_start_docstrings(
    "The bare CLS Model outputting raw hidden-states without any specific head on top.",
    CLS_START_DOCSTRING,
)
class CLSPreTrainedModel(PreTrainedModel):
    config_class = CLSConfig
    base_model_prefix = "model"

    def _init_weights(self, module):
        std = self.config.initializer_range
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=std)
            if module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.Embedding):
            module.weight.data.normal_(mean=0.0, std=std)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()

class CLS(CLSPreTrainedModel):
    def __init__(
            self, 
            config: CLSConfig,
            **kwargs
            ):
        super().__init__(config)
        self.model_config = AutoConfig.from_pretrained("/data/pretrained-models/meta/Llama-3.2-3B-Instruct")
        self.model = LlamaModel.from_pretrained("/data/pretrained-models/meta/Llama-3.2-3B-Instruct")
        self.model.requires_grad_(False)
        self.model.eval()
        print(self.config)
        self.task_score = nn.Linear(config.hidden_size, config.num_tasks, bias=False, device=self.model.device, dtype=self.model.dtype)
        self.label_score = nn.Linear(config.hidden_size, config.num_labels, bias=False, device=self.model.device, dtype=self.model.dtype)
        self.ignore_index = -100
        self.post_init()

    def forward(
        self,
        input_ids: torch.LongTensor = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_values: Optional[Union[Cache, List[torch.FloatTensor]]] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        type_labels: Optional[torch.LongTensor] = None,
        task_labels: Optional[torch.LongTensor] = None,
        use_cache: Optional[bool] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
        cache_position: Optional[torch.LongTensor] = None,
    ) -> Union[Dict, Tuple, torch.Tensor, CLSOutput]:
        with torch.no_grad():
            if input_ids is not None:
                batch_size = input_ids.shape[0]
            else:
                batch_size = inputs_embeds.shape[0]
            outputs = self.model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                position_ids=position_ids,
                past_key_values=past_key_values,
                inputs_embeds=inputs_embeds,
                use_cache=use_cache,
                output_attentions=output_attentions,
                output_hidden_states=output_hidden_states,
                return_dict=return_dict,
                cache_position=cache_position,
            )
            hidden_states = outputs[0]

        if self.model_config.pad_token_id is None:
            sequence_lengths = -1
        else:
            if input_ids is not None:
                # if no pad token found, use modulo instead of reverse indexing for ONNX compatibility
                sequence_lengths = torch.eq(input_ids, self.model_config.pad_token_id).int().argmax(-1) - 1
                sequence_lengths = sequence_lengths % input_ids.shape[-1]
                sequence_lengths = sequence_lengths.to(hidden_states.device)
            else:
                sequence_lengths = -1

        eos_hidden_states = hidden_states[torch.arange(batch_size, device=hidden_states.device), sequence_lengths]
        label_scores = self.label_score(eos_hidden_states)
        task_scores = self.task_score(eos_hidden_states)
        if type_labels is not None and task_labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(label_scores.float(), type_labels) + loss_fct(task_scores, task_labels)
            return (loss, )
        else:
            label_logits = torch.softmax(label_scores.float(), dim=-1)
            task_scores = torch.softmax(task_scores.float(), dim=-1)
        return (label_logits, task_scores, )
        


In [13]:
mix_domains = [
    {
        "input_ids": tokenizer.encode(item['inputs']),
        'task': item['task_label'],
        'label': item['label'],
    }
    for item in tqdm(mix_domains)
]
# eval_datasets = load_dataset(
#     path="json",
#     data_files="/data/lihz/datasets/mix_domains_eval/mix_domains_eval.jsonl",
# )

100%|██████████| 5796/5796 [00:21<00:00, 268.77it/s] 


In [14]:
print(mix_domains[0])

{'input_ids': [128000, 128006, 882, 128007, 271, 22818, 279, 2768, 3488, 323, 3116, 9322, 11503, 320, 32, 11, 426, 11, 356, 323, 423, 705, 5268, 279, 1888, 4320, 627, 14924, 25, 2650, 1690, 3944, 645, 1587, 264, 5410, 35528, 617, 5380, 32, 13, 832, 198, 33, 13, 1403, 198, 34, 13, 3116, 198, 35, 13, 8223, 198, 7927, 2077, 1288, 842, 449, 330, 791, 1888, 4320, 374, 510, 1820, 29634, 47217, 19727, 1405, 279, 510, 1820, 29634, 47217, 60, 374, 832, 315, 362, 11, 426, 11, 356, 477, 423, 13, 128009, 128006, 78191, 128007, 271, 791, 1888, 4320, 374, 426, 13, 128009, 128006, 882, 128007, 271, 22818, 279, 2768, 3488, 323, 3116, 9322, 11503, 320, 32, 11, 426, 11, 356, 323, 423, 705, 5268, 279, 1888, 4320, 627, 14924, 25, 3639, 2035, 374, 7086, 304, 279, 2316, 315, 279, 220, 4468, 24, 3974, 8176, 555, 7091, 49428, 37034, 70325, 5380, 32, 13, 70695, 198, 33, 13, 37235, 564, 276, 198, 34, 13, 31930, 49624, 198, 35, 13, 13527, 198, 7927, 2077, 1288, 842, 449, 330, 791, 1888, 4320, 374, 510, 1820, 296

In [15]:
print(mix_domains[0]['input_ids'])

[128000, 128006, 882, 128007, 271, 22818, 279, 2768, 3488, 323, 3116, 9322, 11503, 320, 32, 11, 426, 11, 356, 323, 423, 705, 5268, 279, 1888, 4320, 627, 14924, 25, 2650, 1690, 3944, 645, 1587, 264, 5410, 35528, 617, 5380, 32, 13, 832, 198, 33, 13, 1403, 198, 34, 13, 3116, 198, 35, 13, 8223, 198, 7927, 2077, 1288, 842, 449, 330, 791, 1888, 4320, 374, 510, 1820, 29634, 47217, 19727, 1405, 279, 510, 1820, 29634, 47217, 60, 374, 832, 315, 362, 11, 426, 11, 356, 477, 423, 13, 128009, 128006, 78191, 128007, 271, 791, 1888, 4320, 374, 426, 13, 128009, 128006, 882, 128007, 271, 22818, 279, 2768, 3488, 323, 3116, 9322, 11503, 320, 32, 11, 426, 11, 356, 323, 423, 705, 5268, 279, 1888, 4320, 627, 14924, 25, 3639, 2035, 374, 7086, 304, 279, 2316, 315, 279, 220, 4468, 24, 3974, 8176, 555, 7091, 49428, 37034, 70325, 5380, 32, 13, 70695, 198, 33, 13, 37235, 564, 276, 198, 34, 13, 31930, 49624, 198, 35, 13, 13527, 198, 7927, 2077, 1288, 842, 449, 330, 791, 1888, 4320, 374, 510, 1820, 29634, 47217, 197

In [16]:
model = CLS.from_pretrained("/data/lihz/projects/instruct/IFT/ift/cls_49_v2")
# model.model = LlamaModel.from_pretrained("/data/lihz/projects/instruct/IFT/code/3b-e3")
model = model.to("npu:0")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

CLSConfig {
  "_name_or_path": "/data/lihz/projects/instruct/IFT/ift/cls_49_v2",
  "architectures": [
    "CLS"
  ],
  "hidden_size": 3072,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2",
    "3": "LABEL_3"
  },
  "initializer_range": 0.02,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2,
    "LABEL_3": 3
  },
  "model_type": "cls",
  "num_tasks": 9,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.43.2"
}



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [17]:
# data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
# data_collator(mix_domains[-1])

tag = 30
with torch.no_grad():
    outputs = model(
        input_ids=torch.tensor([mix_domains[tag]['input_ids']], device=model.device),
        use_cache=False,
    )

In [18]:
print(tokenizer.decode(mix_domains[tag]['input_ids']))
# print(mix_domains[tag]['input_ids'])


<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Given the following question and four candidate answers (A, B, C and D), choose the best answer.
Question: Which of the following did the post-war welfare state of 1948 not aim to provide:
A. free health care and education for all
B. a minimum wage
C. full employment
D. universal welfare
Your response should end with "The best answer is [the_answer_letter]" where the [the_answer_letter] is one of A, B, C or D.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The best answer is B.<|eot_id|><|start_header_id|>user<|end_header_id|>

Given the following question and four candidate answers (A, B, C and D), choose the best answer.
Question: What does Berger (1963) describe as a metaphor for social reality?
A. a fairground ride
B. a circus
C. a puppet theatre
D. a ballet
Your response should end with "The best answer is [the_answer_letter]" where the [the_answer_letter] is one of A, B, C or D.<|eot_id|><|start_header_id|>assist

In [19]:
print(mix_domains[tag]['label'])
print(mix_domains[tag]['task'])

1
0


In [20]:
print(outputs[0][0].cpu())
print(outputs[1][0].cpu())

tensor([0.0124, 0.9597, 0.0103, 0.0176])
tensor([8.8451e-01, 2.1920e-02, 6.8573e-02, 1.1078e-03, 1.4091e-03, 2.6708e-03,
        7.1045e-03, 3.7248e-04, 1.2332e-02])


In [21]:
labels = [0, 0, 0, 0]
idx2labels = ['general', 'reason', 'math', 'code']
tasks = [0, 0, 0, 0, 0, 0, 0, 0, 0]
# idx2tasks = ['mmlu', 'ultrachat', 'gpqa', 'arc_c', 'gsm8k', 'mathinstruct', 'math', 'humaneval', 'mbpp', 'magicoder']
idx2tasks = ['mmlu', 'ultrachat', 'gpqa', 'arc_c', 'gsm8k', 'math', 'humaneval', 'mbpp', 'magicoder']
results = {
    "labels": {
        "general": 0,
        "reason": 0,
        "math": 0,
        "code": 0,
    },
    "tasks": {
        "mmlu":0,
        "ultrachat":0,
        "gpqa":0,
        "arc_c":0,
        "gsm8k":0,
        # "mathinstruct":0,
        "math":0,
        "humaneval":0,
        "mbpp":0,
        "magicoder":0,
    }
}
with torch.no_grad():
    for item in tqdm(mix_domains):
        outputs = model(
            input_ids=torch.tensor([item['input_ids']], device=model.device),
            use_cache=False,
        )
        labels[item['label']] += 1
        tasks[item['task']] += 1
        results['labels'][idx2labels[item['label']]] += (outputs[0][0].argmax(dim=-1).cpu().item() == item['label'])
        results['tasks'][idx2tasks[item['task']]] += (outputs[1][0].argmax(dim=-1).cpu().item() == item['task'])
for i, k in enumerate(idx2labels):
    results['labels'][k] = results['labels'][k] / labels[i]
for i, k in enumerate(idx2tasks):
    results['tasks'][k] = results['tasks'][k] / tasks[i]

  0%|          | 0/5796 [00:00<?, ?it/s]

100%|██████████| 5796/5796 [17:47<00:00,  5.43it/s]


In [22]:
print(labels)
print(tasks)
def print_dict(d, indent=0):
    for key, value in d.items():
        if isinstance(value, dict):
            print(" " * indent + f"{key}:")
            print_dict(value, indent + 4)  # 增加缩进
        else:
            print(" " * indent + f"{key}: {value}")

print_dict(results)


[1099, 1310, 2198, 1189]
[1099, 1099, 47, 164, 1099, 1099, 13, 77, 1099]
labels:
    general: 0.7424931756141947
    reason: 1.0
    math: 0.9940855323020928
    code: 0.7005887300252313
tasks:
    mmlu: 0.9981801637852593
    ultrachat: 0.9044585987261147
    gpqa: 0.0
    arc_c: 0.0
    gsm8k: 0.9945404913557779
    math: 1.0
    humaneval: 0.0
    mbpp: 0.0
    magicoder: 0.272975432211101


In [ ]:
# labels:
#     general: 0.7424931756141947
#     reason: 1.0
#     math: 0.9940855323020928
#     code: 0.7005887300252313
# tasks:
#     mmlu: 0.9981801637852593
#     ultrachat: 0.9044585987261147
#     gpqa: 0.0
#     arc_c: 0.0
#     gsm8k: 0.9945404913557779
#     math: 1.0
#     humaneval: 0.0
#     mbpp: 0.0
#     magicoder: 0.272975432211101

In [23]:
# labels:
#     general: 0.8383838383838383
#     reason: 1.0
#     math: 0.9848484848484849
#     code: 0.5873015873015873
# tasks:
#     mmlu: 0.9393939393939394
#     ultrachat: 0.696969696969697
#     gpqa: 0.0
#     arc_c: 0.0
#     gsm8k: 0.9696969696969697
#     math: 0.8888888888888888
#     humaneval: 0.07692307692307693
#     mbpp: 0.0
#     magicoder: 0.35353535353535354
# labels:
#     general: 0.898989898989899
#     reason: 1.0
#     math: 1.0
#     code: 0.656084656084656
# tasks:
#     mmlu: 1.0
#     ultrachat: 0.98989898989899
#     gpqa: 0.0
#     arc_c: 0.0
#     gsm8k: 1.0
#     math: 1.0
#     humaneval: 0.07692307692307693
#     mbpp: 0.0
#     magicoder: 0.696969696969697

In [24]:

# labels:
#     general: 0.7878787878787878
#     reason: 0.9903225806451613
#     math: 0.6576715497301465
#     code: 0.5714285714285714
# tasks:
#     mmlu: 0.9393939393939394
#     ultrachat: 0.6666666666666666
#     gpqa: 0.0
#     arc_c: 0.0
#     gsm8k: 0.9797979797979798
#     mathinstruct: 0.006369426751592357
#     math: 1.0
#     humaneval: 0.0
#     mbpp: 0.05194805194805195
#     magicoder: 0.2828282828282828

In [25]:
# labels:
#     general: 0.898989898989899
#     reason: 1.0
#     math: 0.2582883577486507
#     code: 0.6931216931216931
# tasks:
#     mmlu: 1.0
#     ultrachat: 0.9696969696969697
#     gpqa: 0.0
#     arc_c: 0.0
#     gsm8k: 1.0
#     mathinstruct: 0.0
#     math: 1.0
#     humaneval: 0.0
#     mbpp: 0.03896103896103896
#     magicoder: 0.47474747474747475
# labels:
#     general: 0.7474747474747475
#     reason: 1.0
#     math: 0.3600616808018504
#     code: 0.6084656084656085
# tasks:
#     mmlu: 1.0
#     ultrachat: 0.9292929292929293
#     gpqa: 0.0
#     arc_c: 0.0
#     gsm8k: 0.98989898989899
#     mathinstruct: 0.0
#     math: 1.0
#     humaneval: 0.0
#     mbpp: 0.012987012987012988
#     magicoder: 0.23232323232323232
# labels:
#     general: 0.9494949494949495
#     reason: 0.0
#     math: 0.24518118735543562
#     code: 0.6507936507936508
# tasks:
#     mmlu: 1.0
#     ultrachat: 0.9696969696969697
#     gpqa: 0.0
#     arc_c: 0.0
#     gsm8k: 1.0
#     mathinstruct: 0.0
#     math: 1.0
#     humaneval: 0.0
#     mbpp: 0.03896103896103896
#     magicoder: 0.46464646464646464

In [26]:
# gsm8k = load_dataset(
#     "gsm8k",
#     'main'
# )
# print(gsm8k['train'][0]['question'])
# print(gsm8k['train'][0]['answer'])

In [27]:
print(math_datasets[0][3]['inst'])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Given the following problem, reason and give a final answer to the problem.
Problem: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?
Your response should end with "The final answer is [answer]" where [answer] is the response to the problem.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The final answer is 6<|eot_id|><|start_header_id|>user<|end_header_id|>

Given the following problem, reason and give a final answer to the problem.
Problem: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?
Your response 